---
## Google Colab Setup

> **Skip this section if you are running locally.** The cell below detects whether
> you are on Google Colab and, if so, installs all dependencies automatically.
> On a local machine with the environment already set up, this cell does nothing.

The setup cell will:
1. Clone the Prometheus repository into `/content/prometheus`
2. Install all Python dependencies via pip
3. Compile the CPU PPC photon-propagation binary from source (requires only g++, which Colab provides)

**PROPOSAL physics tables** are built automatically on first use and cached in
`resources/proposal_tables/` inside the cloned repo. Because Colab sessions are
ephemeral, the tables must be rebuilt each session (~5–10 minutes). To avoid this,
mount a Google Drive folder and point `PROPOSAL_tables_path` in the config to a
persistent path on your Drive (see the optional cell below).


In [ ]:
import sys
import os
import subprocess

ON_COLAB = "google.colab" in sys.modules

# Base URL for pre-computed tutorial assets (GitHub Release)
TUTORIAL_RELEASE_BASE = (
    "https://github.com/jlazar17/prometheus-tutorial/releases/download/tutorial-v1"
)

if ON_COLAB:
    REPO = "/content/prometheus"

    # 1. Clone the repository
    if not os.path.isdir(REPO):
        print("Cloning Prometheus (tutorial-compat branch)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "tutorial-compat",
             "https://github.com/Harvard-Neutrino/prometheus.git", REPO],
            check=True,
        )
        print("  -> cloned")
    else:
        print("Prometheus repo already present, pulling latest...")
        subprocess.run(["git", "pull"], cwd=REPO, check=True)
        print("  -> up to date")

    # 2. Install Python dependencies
    print("Installing Python dependencies (this may take a minute)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "leptoninjector", "proposal", "LeptonWeighter",
         "awkward", "pyarrow", "matplotlib"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO],
        check=True,
    )
    print("  -> dependencies installed")

    # 3. Compile CPU PPC binary from source (g++ is available on Colab)
    ppc_src = os.path.join(REPO, "resources", "PPC_executables", "PPC")
    ppc_bin = os.path.join(ppc_src, "ppc")
    if not os.path.exists(ppc_bin):
        print("Compiling PPC photon propagator (CPU)...")
        subprocess.run(["make", "cpu"], cwd=ppc_src, check=True)
        print("  -> compiled")
    else:
        print("PPC binary already compiled, skipping.")

    # 4. Download PROPOSAL physics tables (avoids ~8 min rebuild per session)
    import urllib.request, tarfile
    tables_dir = os.path.join(REPO, "resources", "PROPOSAL_tables")
    # Check for actual .dat files, not just the directory or a stale subdirectory
    dat_files = [f for f in os.listdir(tables_dir) if f.endswith(".dat")] if os.path.isdir(tables_dir) else []
    if not dat_files:
        print("Downloading PROPOSAL tables (~24 MB)...")
        tables_tar = "/tmp/proposal_tables.tar.gz"
        urllib.request.urlretrieve(
            f"{TUTORIAL_RELEASE_BASE}/proposal_tables.tar.gz", tables_tar
        )
        with tarfile.open(tables_tar) as tf:
            tf.extractall(os.path.join(REPO, "resources"))
        dat_files = [f for f in os.listdir(tables_dir) if f.endswith(".dat")]
        print(f"  -> {len(dat_files)} table files extracted")
    else:
        print(f"PROPOSAL tables already present ({len(dat_files)} files), skipping.")

    # 5. Download pre-computed simulation output and LI config files
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    import prometheus as _p
    _output_dir = os.path.join(
        os.path.abspath(os.path.join(os.path.dirname(_p.__file__), "..")),
        "tutorial_output",
    )
    os.makedirs(_output_dir, exist_ok=True)
    # tutorial_water.parquet is optional — skip gracefully if not on the release
    _required = {"tutorial_muon_cc.parquet", "tutorial_nue_cc.parquet",
                 "1211_LI_config.lic", "925_LI_config.lic"}
    _optional = {"tutorial_water.parquet"}
    for _fname in [
        "tutorial_muon_cc.parquet",
        "tutorial_nue_cc.parquet",
        "tutorial_water.parquet",
        "1211_LI_config.lic",
        "925_LI_config.lic",
    ]:
        _dest = os.path.join(_output_dir, _fname)
        if not os.path.exists(_dest):
            print(f"Downloading {_fname}...")
            try:
                urllib.request.urlretrieve(f"{TUTORIAL_RELEASE_BASE}/{_fname}", _dest)
                print(f"  -> saved to {_dest}")
            except Exception as _e:
                if _fname in _optional:
                    print(f"  -> not available on release, skipping ({_e})")
                else:
                    raise

    print("\nColab setup complete.")
else:
    print("Not on Colab — skipping setup.")


### Optional: persist PROPOSAL tables across sessions (Google Drive)

Run the cell below if you want to avoid rebuilding PROPOSAL tables every session.
Skip it if you are happy to wait ~5–10 minutes on first use each session.


In [ ]:
# Optional: mount Google Drive and cache PROPOSAL tables there.
# Uncomment and run this cell before the simulation cells.

# if ON_COLAB:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     PROPOSAL_CACHE = "/content/drive/MyDrive/prometheus_proposal_tables"
#     import os; os.makedirs(PROPOSAL_CACHE, exist_ok=True)
#     # Symlink so prometheus finds tables in the expected location
#     import prometheus as _p
#     table_target = os.path.join(os.path.dirname(_p.__file__), "..", "resources", "proposal_tables")
#     if not os.path.exists(table_target):
#         os.symlink(PROPOSAL_CACHE, table_target)
#     print(f"PROPOSAL tables will be cached at: {PROPOSAL_CACHE}")


# Prometheus Neutrino Telescope Simulation Tutorial

## What is Prometheus?

Prometheus is an end-to-end Monte Carlo simulation framework for neutrino telescopes. Given a detector geometry and a set of optical medium properties, Prometheus simulates the full chain from neutrino injection through final photon detection:

1. **Neutrino injection** (via LeptonInjector): generates neutrino interaction events distributed throughout a fiducial volume around the detector according to specified energy and angular spectra.
2. **Lepton propagation** (via PROPOSAL): propagates the charged leptons produced in the interaction, computing continuous and stochastic energy losses along the track.
3. **Photon propagation** (via PPC for ice, or Olympus for water): propagates the Cherenkov photons from all energy loss positions to the optical modules, recording arrival times and positions.

The output is a Parquet file containing the Monte Carlo truth and the photon hit record for each simulated event.

## What this tutorial covers

- Running a basic simulation
- Inspecting and understanding the output format
- Visualizing simulated events in the detector
- Plotting Monte Carlo truth distributions
- Computing the effective area using the oneweighting formalism
- Reweighting to physics fluxes and comparing particle types
- Building custom PPC optical property tables for a new medium (e.g., Mediterranean deep water)

**Prerequisites:** The Prometheus package and its dependencies must be installed. All cells are designed to run top-to-bottom in a fresh kernel.

---
## Section 1: Introduction and Setup

We begin by importing standard libraries and defining paths. All file paths are constructed from the repository root so the notebook is portable.

`REPO_DIR` is determined dynamically from the `prometheus` package location so it works regardless of where you cloned the repository.

In [ ]:
import os
# Suppress HDF5 version mismatch warnings from LeptonInjector
os.environ["HDF5_DISABLE_VERSION_CHECK"] = "1"

import sys
import copy
import shutil
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import awkward as ak
import pyarrow.parquet as pq

# Determine repository and resource directories from the installed package location
import prometheus as _prom_pkg
REPO_DIR     = os.path.abspath(os.path.join(os.path.dirname(_prom_pkg.__file__), ".."))
RESOURCE_DIR = os.path.join(REPO_DIR, "resources")

# All simulation output for this tutorial goes here
OUTPUT_DIR = os.path.join(REPO_DIR, "tutorial_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Repository root : {REPO_DIR}")
print(f"Resource directory: {RESOURCE_DIR}")
print(f"Output directory  : {OUTPUT_DIR}")

In [ ]:
# Convenience paths that are referenced throughout the tutorial
GEO_FILE_ICE    = os.path.join(RESOURCE_DIR, "geofiles", "icecube.geo")
XSEC_DIR        = os.path.join(RESOURCE_DIR, "cross_section_splines")
PPC_EXE         = os.path.join(RESOURCE_DIR, "PPC_executables", "PPC", "ppc")
PPC_TABLES_SP   = os.path.join(RESOURCE_DIR, "PPC_tables", "south_pole")
PPC_TMPDIR      = os.path.join(OUTPUT_DIR, "ppc_tmp")

# Verify critical files exist
for label, path in [
    ("icecube.geo", GEO_FILE_ICE),
    ("cross section splines", XSEC_DIR),
    ("PPC executable", PPC_EXE),
    ("south_pole PPC tables", PPC_TABLES_SP),
]:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  {status}  {label}: {path}")

---
## Section 2: Running a Simulation

### The simulation pipeline

Each Prometheus run executes three stages in sequence:

**Stage 1 — Neutrino injection (LeptonInjector)**  
LeptonInjector samples neutrino interactions from a power-law energy spectrum within a specified zenith/azimuth range. For each event it records the neutrino kinematics, the interaction type (CC=1, NC=2), Bjorken-x/y, and the positions of the final-state particles relative to the detector. It also writes a `.lic` (LeptonInjector Configuration) file that encodes the generation weights needed for later flux reweighting.

**Stage 2 — Lepton propagation (PROPOSAL)**  
The charged lepton (muon, tau, or electron) is propagated through the medium using the PROPOSAL code. PROPOSAL models all relevant energy-loss processes: ionisation, bremsstrahlung, photo-nuclear interactions, and pair production. Stochastic losses above a threshold are treated as separate "sub-showers", each contributing Cherenkov photons.

**Stage 3 — Photon propagation (PPC for ice)**  
The Photon Propagation Code (PPC) propagates individual Cherenkov photons from every energy-loss point to the optical modules, accounting for scattering and absorption in the ice (or water) according to the optical property tables. PPC records the photon arrival time and the hit DOM for each detected photon.

### The `config` object

Prometheus is configured via a module-level dictionary `config`. Because it is a global object, modifications persist across cells and across `Prometheus` instantiations in the same Python session. In this tutorial we always build the config explicitly from scratch to avoid surprises.

In [ ]:
# --- Re-injection: electron neutrino CC simulation ---
#
# Electron neutrinos produce electrons in CC interactions. Because an electron
# stops within centimetres (electromagnetic cascade), the entire energy is
# deposited in a compact sphere. This requires *volume injection*: the vertex
# is sampled uniformly inside the detector fiducial volume rather than being
# placed far upstream like a through-going muon.

import importlib
import prometheus as prom_module
importlib.reload(prom_module)
from prometheus import Prometheus, config

_NUE_OUTFILE = os.path.join(OUTPUT_DIR, "my_nue_cc.parquet")

if os.path.exists(_NUE_OUTFILE):
    print(f"Pre-computed output found: {_NUE_OUTFILE}")
    print("Skipping simulation — delete the file to re-run.")
else:
    config["run"]["nevents"]           = 3
    config["run"]["random state seed"] = 99
    config["run"]["outfile"]          = _NUE_OUTFILE
    config["run"]["storage prefix"]   = OUTPUT_DIR

    config["detector"]["geo file"] = GEO_FILE_ICE

    config["injection"]["LeptonInjector"]["paths"]["xsec dir"]               = XSEC_DIR
    config["injection"]["LeptonInjector"]["simulation"]["final state 1"]     = "EMinus"
    config["injection"]["LeptonInjector"]["simulation"]["final state 2"]     = "Hadrons"
    config["injection"]["LeptonInjector"]["simulation"]["minimal energy"]    = 1e3
    config["injection"]["LeptonInjector"]["simulation"]["maximal energy"]    = 1e4
    config["injection"]["LeptonInjector"]["simulation"]["is ranged"]         = False

    config["photon propagator"]["name"]                                = "PPC"
    config["photon propagator"]["PPC"]["paths"]["ppc_exe"]             = PPC_EXE
    config["photon propagator"]["PPC"]["paths"]["ppctables"]           = PPC_TABLES_SP
    config["photon propagator"]["PPC"]["paths"]["ppc_tmpdir"]          = PPC_TMPDIR
    config["photon propagator"]["PPC"]["paths"]["force"]               = True

    print("Running nu_e CC simulation (3 events, volume injection)...")
    p_nue = Prometheus(userconfig=config)
    p_nue.sim()
    print(f"NuE simulation complete.")

print(f"Output: {_NUE_OUTFILE}")

# Use simulation output if available, otherwise fall back to pre-computed
_my_nue = os.path.join(OUTPUT_DIR, "my_nue_cc.parquet")
_precomputed_nue = os.path.join(OUTPUT_DIR, "tutorial_nue_cc.parquet")
NUE_PARQUET = _precomputed_nue
NUMU_PARQUET = os.path.join(OUTPUT_DIR, "tutorial_muon_cc.parquet")
print(f"NUE_PARQUET -> {NUE_PARQUET}")

--- 
## Adapting the Config for a Muon Simulation

The nu_e CC simulation above uses **volume injection** because the electron cascade is fully contained inside the detector. Simulating nu_mu CC (muon neutrino charged-current) interactions requires a few config changes:

**1. Change the injection name** (if using the `injection.name` key):
```
config["injection"]["name"] = "LeptonInjector"
```

**2. Change the final state to a muon:**
```python
config["injection"]["LeptonInjector"]["simulation"]["final state 1"] = "MuMinus"
```

**3. Switch to ranged injection:**
```python
config["injection"]["LeptonInjector"]["simulation"]["is ranged"] = True
```

**Why ranged injection?**  
A muon produced in a CC interaction has a range of hundreds of metres to kilometres (depending on energy). The muon can enter the detector from outside, so the interaction vertex should be sampled *upstream* in the rock or ice — not uniformly inside the fiducial volume. Ranged injection places the vertex at the appropriate distance before the detector boundary, allowing the muon to travel through material and lose energy before reaching the instrumented volume.

**Topology difference:**  
- **nu_e CC (this tutorial)** — the electron immediately showers into an electromagnetic cascade. All the light is deposited in a compact sphere (~10 m). This is called a *cascade* or *shower* topology.  
- **nu_mu CC** — the muon travels tens to hundreds of metres, producing a Cherenkov cone along its track. This is called a *track* topology, and produces a very different hit pattern: hits are spread across many DOMs along the track direction rather than clustered near a single vertex.


---
## Section 3: Examining the Output

### Parquet and Awkward Arrays

Prometheus writes its output in Apache Parquet format, a columnar binary format designed for large datasets. The file is read back using the **awkward-array** library, which provides NumPy-like operations on arrays of variable-length lists — essential here because different events have different numbers of photon hits.

The top-level structure is:

```
data[i]  ← the i-th event
  .mc_truth  ← scalar truth quantities for this event
  .photons   ← variable-length arrays of photon hit records
```

The `mc_truth` record stores both the initial neutrino state (energy, direction, vertex) and the full list of final-state particles (the lepton and the hadronic shower, plus any secondary particles produced during propagation).

The `photons` record stores one entry per detected photon: which DOM was hit, the arrival time in nanoseconds, and the position of that DOM.

Importantly, the simulation configuration is stored as JSON in the Parquet file metadata, so the output is self-describing.

In [ ]:
# Load the simulation output
data = ak.from_parquet(NUE_PARQUET)

print(f"Number of events: {len(data)}")
print()
print("Top-level fields:")
for field in data.fields:
    print(f"  {field}")
print()
print("mc_truth fields:")
for field in data["mc_truth"].fields:
    print(f"  {field}")
print()
if "photons" in data.fields:
    print("photons fields:")
    for field in data["photons"].fields:
        print(f"  {field}")
else:
    print("photons: (no hits recorded in this dataset)")

In [ ]:
# Inspect the metadata stored in the Parquet file
import json
meta = pq.read_metadata(NUE_PARQUET)
stored_config = json.loads(meta.metadata[b"config_prometheus"])
print("Stored simulation config keys:", list(stored_config.keys()))
print("Injector:", stored_config["injection"]["name"])
print("n events:", stored_config["run"]["nevents"])

In [ ]:
# Inspect the Monte Carlo truth for a single event
event_idx = 0
event = data[event_idx]
truth = event["mc_truth"]

print(f"=== Event {event_idx} MC Truth ===")
print(f"  Interaction type     : {int(truth['interaction'])}  (1=CC, 2=NC)")
print(f"  Initial neutrino energy: {float(truth['initial_state_energy']):.1f} GeV")
print(f"  Initial state PDG type : {int(truth['initial_state_type'])}  (14 = nu_mu)")
print(f"  Zenith  (rad)          : {float(truth['initial_state_zenith']):.4f}")
print(f"  Azimuth (rad)          : {float(truth['initial_state_azimuth']):.4f}")
print(f"  Vertex (x,y,z) m       : ({float(truth['initial_state_x']):.1f}, "
      f"{float(truth['initial_state_y']):.1f}, {float(truth['initial_state_z']):.1f})")
print(f"  Bjorken x              : {float(truth['bjorken_x']):.4f}")
print(f"  Bjorken y              : {float(truth['bjorken_y']):.4f}")
print()
print("Final-state particles:")
for i, (pdg, e, parent) in enumerate(zip(
    truth["final_state_type"],
    truth["final_state_energy"],
    truth["final_state_parent"]
)):
    print(f"  [{i}] PDG={int(pdg):12d}  E={float(e):10.2f} GeV  parent_idx={int(parent)}")

In [ ]:
# Count photon hits per event
nhits = ak.num(data["photons"]["t"]).to_numpy()

print(f"Total photon hits across all events: {nhits.sum():,}")
print(f"Mean hits per event : {nhits.mean():.1f}")
print(f"Median hits per event: {np.median(nhits):.1f}")
print(f"Max hits in one event: {nhits.max():,}")
print(f"Events with zero hits: {(nhits == 0).sum()} / {len(nhits)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
ax = axes[0]
ax.hist(nhits, bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Number of photon hits")
ax.set_ylabel("Events")
ax.set_title("Hit multiplicity distribution (linear)")

# Log scale (excluding zero-hit events)
ax = axes[1]
nonzero = nhits[nhits > 0]
bins = np.logspace(np.log10(nonzero.min()), np.log10(nonzero.max()), 25)
ax.hist(nonzero, bins=bins, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xscale("log")
ax.set_xlabel("Number of photon hits")
ax.set_ylabel("Events")
ax.set_title("Hit multiplicity distribution (log scale, non-zero events)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hit_multiplicity.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Summarize which events have hits and which do not
has_hits = nhits > 0
print(f"Events with at least 1 hit: {has_hits.sum()}")
print(f"Events with zero hits      : {(~has_hits).sum()}")
print()
print("Zero-hit events tend to occur when the neutrino vertex is far from the detector")
print("or the muon exits the instrumented volume before producing detectable light.")
print()
# Show the energy distribution of zero-hit vs hit events
energies = ak.to_numpy(data["mc_truth"]["initial_state_energy"])
bins = np.logspace(np.log10(energies.min()), np.log10(energies.max()), 10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(
    [energies[has_hits], energies[~has_hits]],
    bins=bins,
    stacked=True,
    label=["Hit events", "Zero-hit events"],
    color=["steelblue", "firebrick"],
)
ax.set_xscale("log")
ax.set_xlabel("Neutrino energy [GeV]")
ax.set_ylabel("Events")
ax.set_title("Energy distribution: hit vs zero-hit events")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hit_vs_nohit_energy.pdf"), bbox_inches="tight")
plt.show()

---
## Section 4: Visualizing Events in the Detector

### Event display

Prometheus provides `plot_event` and `plot_brightest` for 3-D event visualizations. Each hit DOM is shown as a colored dot:

- **Size** scales logarithmically with the number of photon hits on that DOM.
- **Color** encodes the mean arrival time of photons on that DOM, from early (one end of the colormap) to late (the other end).
- Grey dots in the background are all DOMs in the detector.

The Cherenkov cone geometry is visible as a ring of early-hit DOMs around the muon track.

In [ ]:
from prometheus.detector import detector_from_geo
from prometheus.plotting import plot_brightest, plot_event

# Load detector geometry
detector = detector_from_geo(GEO_FILE_ICE)
print(f"Detector loaded: {detector.n_modules} optical modules")
print(f"Detector center offset (x, y, z): {detector.offset}")

In [ ]:
# Plot the event with the most photon hits and save it to a file
brightest_event_idx = np.argmax(nhits)
print(f"Brightest event index: {brightest_event_idx}  ({nhits[brightest_event_idx]} hits)")

DISPLAY_FILE = os.path.join(OUTPUT_DIR, "event_display_brightest.pdf")

plot_brightest(
    data,
    detector,
    figname=DISPLAY_FILE,
    save=True,
    show=True,
    channel="photons",
    show_doms=True,
    show_lepton=True,
    charge_mult=5,
    cmap="jet_r",
    tmethod="min"
)

In [ ]:
muondata = ak.from_parquet(NUMU_PARQUET)

DISPLAY_FILE = os.path.join(OUTPUT_DIR, "event_display_brightest.pdf")

plot_event(
    muondata[445],
    detector,
    save=False,
    show=True,
    channel="photons",
    show_doms=True,
    show_lepton=True,
    charge_mult=5,
    cmap="jet_r",
    tmethod="min"
)

In [ ]:
# Plot the event with the most photon hits and save it to a file
brightest_event_idx = np.argmax(nhits)
print(f"Brightest event index: {brightest_event_idx}  ({nhits[brightest_event_idx]} hits)")

DISPLAY_FILE = os.path.join(OUTPUT_DIR, "event_display_brightest.pdf")

plot_brightest(
    data,
    detector,
    figname=DISPLAY_FILE,
    save=True,
    show=True,
    channel="photons",
    show_doms=True,
    show_lepton=True,
    charge_mult=5,
    cmap="jet_r",
    tmethod="min"
)# Arrival time histogram for the brightest event
best_event = data[brightest_event_idx]
times = ak.to_numpy(best_event["photons"]["t"])
times = times[times > -1]   # sentinel value -1 means no hit

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(times, bins=60, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Photon arrival time [ns]")
ax.set_ylabel("Photon hits")
ax.set_title(f"Photon arrival time distribution — event {brightest_event_idx}")
ax.axvline(np.median(times), color="firebrick", linestyle="--", label=f"Median = {np.median(times):.0f} ns")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "arrival_time_brightest.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# 2D scatter of hit positions (x-z plane) colored by photon arrival time
# This is the "side view" of the event and reveals the Cherenkov cone geometry.

hx = ak.to_numpy(best_event["photons"]["sensor_pos_x"])
hz = ak.to_numpy(best_event["photons"]["sensor_pos_z"])
ht = ak.to_numpy(best_event["photons"]["t"])

# Mask out sentinel values
valid = ht > -1
hx, hz, ht = hx[valid], hz[valid], ht[valid]

fig, ax = plt.subplots(figsize=(7, 6))

# Show all DOMs in grey
dom_xy = detector.module_coords
ax.scatter(dom_xy[:, 0], dom_xy[:, 2], c="lightgrey", s=1, zorder=1, label="All DOMs")

# Color hits by time
sc = ax.scatter(hx, hz, c=ht, cmap="plasma", s=10, zorder=2, label="Hit DOMs")
plt.colorbar(sc, ax=ax, label="Photon arrival time [ns]")

# Mark the interaction vertex
vx  = float(best_event["mc_truth"]["initial_state_x"])
vz  = float(best_event["mc_truth"]["initial_state_z"])
ax.scatter([vx], [vz], marker="*", s=300, c="gold", zorder=5, label="Vertex")

ax.set_xlabel("x [m]")
ax.set_ylabel("z [m]")
ax.set_title(f"Hit positions (x-z plane) — event {brightest_event_idx}")
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hit_positions_xz.pdf"), bbox_inches="tight")
plt.show()

---
## Section 5: MC Truth Distributions

### What is stored in `mc_truth`?

The `mc_truth` record contains the complete "ground truth" information for each event as it comes out of the Monte Carlo generators. Key quantities include:

| Field | Description |
|---|---|
| `initial_state_type` | PDG code of the incoming neutrino (14 = nu_mu, -14 = anti-nu_mu, 16 = nu_tau, ...) |
| `initial_state_energy` | True neutrino energy [GeV] |
| `initial_state_zenith/azimuth` | Direction of the incoming neutrino [radians] |
| `initial_state_x/y/z` | Neutrino interaction vertex [m] |
| `interaction` | Interaction type (1 = CC, 2 = NC) |
| `bjorken_x`, `bjorken_y` | Deep-inelastic scattering kinematic variables |
| `final_state_type` | PDG code of each final-state particle |
| `final_state_energy` | Energy of each final-state particle [GeV] |
| `final_state_parent` | Index of the parent particle (0 = primary) |

The final-state list includes both the primary particles (parent=0) and any secondary particles produced during PROPOSAL propagation.

Note that because LeptonInjector samples from a power-law spectrum (default E^-1 for unweighted generation), the raw distributions are NOT the same as expected physics fluxes. The weighting section (Section 6) shows how to correct for this.

In [ ]:
# Extract MC truth arrays for all events
truth = data["mc_truth"]

energies  = ak.to_numpy(truth["initial_state_energy"])   # GeV
zeniths   = ak.to_numpy(truth["initial_state_zenith"])   # rad
xs        = ak.to_numpy(truth["initial_state_x"])        # m
ys        = ak.to_numpy(truth["initial_state_y"])        # m
zs        = ak.to_numpy(truth["initial_state_z"])        # m
bx        = ak.to_numpy(truth["bjorken_x"])
by        = ak.to_numpy(truth["bjorken_y"])

print(f"Energy range: {energies.min():.1f} – {energies.max():.1f} GeV")
print(f"Zenith range: {np.degrees(zeniths.min()):.1f}° – {np.degrees(zeniths.max()):.1f}°")

In [ ]:
# Energy spectrum (log-log histogram)
# The spectrum follows E^-1 because LeptonInjector was set to power_law=1.0
# (the default). This is a flat distribution in log(E).

fig, ax = plt.subplots(figsize=(7, 5))
bins = np.logspace(np.log10(energies.min()), np.log10(energies.max()), 20)
counts, edges = np.histogram(energies, bins=bins)
centers = np.sqrt(edges[:-1] * edges[1:])
widths  = np.diff(edges)

# Plot dN/dE to make the power-law index visible
ax.stairs(counts / widths, edges, fill=True, color="steelblue", alpha=0.7)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Neutrino energy [GeV]")
ax.set_ylabel("dN / dE  [GeV$^{-1}$]")
ax.set_title("Generated energy spectrum (E$^{-1}$ injection, 50 events)")
# Overlay E^-1 reference line
e_ref = np.logspace(np.log10(energies.min()), np.log10(energies.max()), 100)
norm  = (counts / widths)[np.argmax(counts / widths)] * (centers[np.argmax(counts / widths)])
ax.plot(e_ref, norm / e_ref, "k--", label="$E^{-1}$  reference")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "energy_spectrum.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Cos(zenith) distribution
# An isotropic injection produces a flat distribution in cos(theta).
# Upgoing events have cos(theta) < 0, downgoing have cos(theta) > 0.
cos_zen = np.cos(zeniths)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cos_zen, bins=20, color="mediumpurple", edgecolor="white", linewidth=0.5)
ax.axvline(0, color="k", linestyle="--", alpha=0.5)
ax.text(-0.7, 0, "Upgoing",   ha="center", va="bottom", fontsize=10, color="grey")
ax.text( 0.7, 0, "Downgoing", ha="center", va="bottom", fontsize=10, color="grey")
ax.set_xlabel("cos(zenith)")
ax.set_ylabel("Events")
ax.set_title("Zenith angle distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cos_zenith.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# 2D vertex position scatter: x-y plane and depth distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# x-y plane (top view)
ax = axes[0]
sc = ax.scatter(xs, ys, c=zs, cmap="viridis", s=20, alpha=0.8)
plt.colorbar(sc, ax=ax, label="z (depth) [m]")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Interaction vertices — top view (x-y)")
ax.set_aspect("equal")

# Depth (z) distribution
ax = axes[1]
ax.hist(zs, bins=20, orientation="horizontal", color="steelblue",
        edgecolor="white", linewidth=0.5)
ax.set_xlabel("Events")
ax.set_ylabel("z (depth) [m]")
ax.set_title("Interaction vertex depth distribution")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "vertex_positions.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
def plot_vertex_rz(xs, ys, zs, zeniths, nhits, ax=None, cmap="plasma"):
    """
    Plot interaction vertices in the r-z plane, colored by zenith angle.
    Events with photon hits are shown as filled markers; zero-hit events as open markers.

    Parameters
    ----------
    xs, ys, zs : arrays of vertex positions [m]
    zeniths     : array of neutrino zenith angles [rad]
    nhits       : array of photon hit counts per event
    ax          : existing matplotlib Axes to draw on (creates new figure if None)
    cmap        : colormap name for zenith angle coloring
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))
    else:
        fig = ax.get_figure()

    r = np.sqrt(xs**2 + ys**2)
    has_hits = nhits > 0
    zen_deg = np.degrees(zeniths)
    vmin, vmax = zen_deg.min(), zen_deg.max()
    cm = plt.get_cmap(cmap)

    def colors(mask):
        return cm((zen_deg[mask] - vmin) / (vmax - vmin))

    # Zero-hit events: open circles
    ax.scatter(
        r[~has_hits], zs[~has_hits],
        c=colors(~has_hits),
        s=15, alpha=0.6,
        facecolors="none",
        edgecolors=colors(~has_hits),
        linewidths=0.8,
        label="No hits",
    )
    # Hit events: filled circles
    sc = ax.scatter(
        r[has_hits], zs[has_hits],
        c=zen_deg[has_hits],
        cmap=cmap, vmin=vmin, vmax=vmax,
        s=15, alpha=0.8,
        label="Has hits",
    )

    plt.colorbar(sc, ax=ax, label="Zenith angle [deg]")
    ax.set_xlabel(r"$r = \sqrt{x^2 + y^2}$ [m]")
    ax.set_ylabel("z [m]")
    ax.set_xlim(0, 1e4)
    ax.set_ylim(None, 0)
    ax.set_title("Interaction vertices — r-z plane")
    ax.legend(loc="upper right", framealpha=0.7)
    plt.tight_layout()
    return fig, ax

truth = muondata["mc_truth"]
zeniths   = ak.to_numpy(truth["initial_state_zenith"])   # rad
xs        = ak.to_numpy(truth["initial_state_x"])        # m
ys        = ak.to_numpy(truth["initial_state_y"])        # m
zs        = ak.to_numpy(truth["initial_state_z"])        # m
bx        = ak.to_numpy(truth["bjorken_x"])
by        = ak.to_numpy(truth["bjorken_y"])
nhitsmuon = ak.num(muondata["photons"]["t"]).to_numpy()

fig, ax = plot_vertex_rz(xs, ys, zs, zeniths, nhitsmuon)
plt.savefig(os.path.join(OUTPUT_DIR, "vertex_rz.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Bjorken x vs y — kinematics of the deep-inelastic scattering
#
# x (Bjorken-x): fraction of the nucleon's momentum carried by the struck quark.
# y (inelasticity): fraction of the neutrino energy transferred to the hadron system.
#   y = 0  →  all energy goes to the lepton (visible as a clean muon track)
#   y = 1  →  all energy goes to the hadron shower

fig, ax = plt.subplots(figsize=(6, 5))
h = ax.hist2d(
    bx, by,
    bins=15,
    range=[[0, 1], [0, 1]],
    cmap="Blues",
    density=False
)
plt.colorbar(h[3], ax=ax, label="Events")
ax.set_xlabel("Bjorken-x")
ax.set_ylabel("Inelasticity y")
ax.set_title("DIS kinematic variables")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "bjorken_xy.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Final-state particle type breakdown
#
# The final-state list includes primary particles (parent=0) plus all secondaries
# produced by PROPOSAL during propagation (e+/e- pairs, bremsstrahlung photons, etc.).
# Here we focus on the primary final states: the lepton and the hadron system.

# PDG code labels
PDG_NAMES = {
    13: r"$\mu^-$",
    -13: r"$\mu^+$",
    15: r"$\tau^-$",
    -15: r"$\tau^+$",
    11: r"$e^-$",
    -11: r"$e^+$",
    -2000001006: "Hadrons",
    2212: "Proton",
    22: r"$\gamma$",
}

# Collect PDG codes of primary final states (parent == 0) across all events
primary_pdg_list = []
for event in data:
    parents = ak.to_numpy(event["mc_truth"]["final_state_parent"])
    pdg_codes = ak.to_numpy(event["mc_truth"]["final_state_type"])
    primary_pdg_list.extend(pdg_codes[parents == 0].tolist())

from collections import Counter
counts = Counter(primary_pdg_list)
labels = [PDG_NAMES.get(k, f"PDG {k}") for k in counts.keys()]
values = list(counts.values())

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values, color=["steelblue", "firebrick", "forestgreen", "darkorange"][:len(labels)])
ax.bar_label(bars)
ax.set_ylabel("Count (across all events)")
ax.set_title("Primary final-state particle types")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "final_state_types.pdf"), bbox_inches="tight")
plt.show()

print("Each CC event contributes exactly one mu- and one Hadron primary final state.")

---
## Section 6: Computing the Effective Area

### The Oneweight Formalism

When LeptonInjector generates events, it does not use the physics flux — instead it samples from a reference distribution (typically a power law in energy, isotropic in direction). To convert raw event counts into physical event rates, each event is assigned an **oneweight** $W_i$ [GeV sr m$^2$] such that:

$$\text{Rate} = \sum_i \frac{W_i \cdot \Phi(E_i, \Omega_i)}{N_\text{gen}}$$

where $\Phi$ is the physical flux [GeV$^{-1}$ sr$^{-1}$ cm$^{-2}$ s$^{-1}$] and $N_\text{gen}$ is the total number of generated events.

### Effective Area

The **neutrino effective area** $A_\text{eff}(E)$ is the equivalent collection area of the detector with 100% efficiency:

$$A_\text{eff}(E) = \frac{1}{\Phi_\text{gen}(E) \cdot \Delta\Omega \cdot \Delta E} \sum_{i: E_i \in [E, E+\Delta E],\, \text{cut}} W_i / N_\text{gen}$$

where the sum runs over events passing the analysis selection cut. The `ParquetWeighter` class reads the `.lic` file path from the Parquet metadata, computes the appropriate LeptonWeighter normalization, and returns one oneweight per event.

In [ ]:
from prometheus.weighting import ParquetWeighter

try:
    weighter = ParquetWeighter(NUE_PARQUET, lic_file=os.path.join(OUTPUT_DIR, "1211_LI_config.lic"))
    oneweights = weighter.weight_events()  # [GeV sr m^2] per event
    print(f"Computed oneweights for {len(oneweights)} events")
    print(f"Oneweight range: {oneweights.min():.3e} – {oneweights.max():.3e} GeV sr m^2")
    print(f"Mean oneweight : {oneweights.mean():.3e} GeV sr m^2")
    WEIGHTING_OK = True
except Exception as exc:
    print(f"WARNING: Could not compute oneweights: {exc}")
    print("The weighting requires LeptonWeighter to be installed and the .lic file to exist.")
    WEIGHTING_OK = False

In [ ]:
if WEIGHTING_OK:
    fig, ax = plt.subplots(figsize=(8, 5))
    MAGIC_CORRECTION = 4
    
    # Apply an analysis selection cut: require at least 5 photon hits
    NHITS_CUT = 60
    for NHITS_CUT in [3, 12, 60]:
        cut_mask = nhits >= NHITS_CUT
    
        print(f"Cut: Nhits >= {NHITS_CUT}")
        print(f"Events passing cut: {cut_mask.sum()} / {len(cut_mask)}")
    
        # Compute effective area in log-spaced energy bins
        # Config: azimuth from 0 to 360 deg, zenith from 0 to 180 deg (4*pi sr)
        min_zen_rad = np.radians(config["injection"]["LeptonInjector"]["simulation"]["min zenith"])
        max_zen_rad = np.radians(config["injection"]["LeptonInjector"]["simulation"]["max zenith"])
        min_az_rad  = np.radians(config["injection"]["LeptonInjector"]["simulation"]["min azimuth"])
        max_az_rad  = np.radians(config["injection"]["LeptonInjector"]["simulation"]["max azimuth"])
    
        delta_omega = (max_az_rad - min_az_rad) * (np.cos(min_zen_rad) - np.cos(max_zen_rad))
        n_total     = config["run"]["nevents"]
    
        energy_bins   = np.logspace(3, 5, 12)
        bin_centers   = np.sqrt(energy_bins[:-1] * energy_bins[1:])
        bin_widths    = np.diff(energy_bins)
        aeff          = np.zeros(len(bin_centers))
    
        for i, (elow, ehigh) in enumerate(zip(energy_bins[:-1], energy_bins[1:])):
            in_bin = (energies >= elow) & (energies < ehigh)
            in_bin_cut = in_bin & cut_mask
            dE = ehigh - elow
            if in_bin.sum() > 0:
                aeff[i] = MAGIC_CORRECTION * oneweights[in_bin_cut].sum() / delta_omega / dE / n_total
    
        # Plot effective area
        
        valid = aeff > 0
        ax.plot(bin_centers[valid], aeff[valid], "o-", lw=2, markersize=6, label=f"Nhits $\geq$ {NHITS_CUT}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Neutrino energy [GeV]")
    ax.set_ylabel(r"$A_{\rm eff}$  [m$^2$]")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "effective_area.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("Skipping effective area calculation (weighting not available).")

---
## Section 7: Custom Injection — Different Physics

### Reweighting vs Re-injection

Once you have a set of simulated events with oneweights, there are two ways to explore different physics assumptions:

**Reweighting:** Apply a new flux $\Phi(E, \Omega)$ to the existing oneweights without re-running the simulation:
$$N_\text{expected} = T \sum_i \frac{W_i \cdot \Phi(E_i, \Omega_i)}{N_\text{gen}}$$
This is fast and covers any flux model, but is limited to the phase-space that was originally generated.

**Re-injection:** Run a completely new simulation with a different particle type, energy range, or angular range. This is necessary when changing the final-state particle type (e.g., muon to electron neutrinos).

Below we demonstrate both approaches.

### Track vs cascade topologies

The two dominant CC neutrino interaction channels produce very different detector signatures:

| Channel | Final state | Topology | Injection mode |
|---|---|---|---|
| $\nu_\mu$ CC | $\mu^-$ + hadrons | **Track**: long muon track through (and beyond) the detector | **Ranged** — vertex placed upstream at the estimated muon range |
| $\nu_e$ CC | $e^-$ + hadrons | **Cascade**: compact EM shower, radius $\lesssim 10$ m | **Volume** — vertex must be inside the detector fiducial volume |

Because an electron deposits virtually all of its energy in a short electromagnetic shower, LeptonInjector uses *volume injection* for $\nu_e$: the interaction vertex is sampled uniformly inside a cylinder enclosing the detector. This is set automatically when `final state 1 = "EMinus"`, but can also be forced explicitly with `is ranged = False`.


In [ ]:
if WEIGHTING_OK:
    # --- Reweighting to an atmospheric neutrino flux proxy ---
    # The conventional atmospheric nu_mu spectrum is approximately E^-3.7.
    # We use a simple power-law proxy: Phi(E) ~ E^-3.7 / (1 + E/E_knee)
    # (the knee suppression is not important here; we use a pure power law for clarity).

    # Normalization of the Honda et al. atmospheric flux at 1 TeV is roughly 1.5e-7
    # in units of [GeV^-1 sr^-1 cm^-2 s^-1], but for illustration we use arbitrary units.

    SPECTRAL_INDEX = 3.7
    E0 = 1e3  # reference energy [GeV]

    def atm_flux_proxy(E):
        """Simple power-law atmospheric neutrino flux proxy [arbitrary units]."""
        return (E / E0) ** (-SPECTRAL_INDEX)

    # Weight per event = W_i * Phi(E_i) / n_total
    # (Multiply by an exposure T [s] and solid angle integration is already in W_i)
    atm_weights = oneweights * atm_flux_proxy(energies) / n_total  # [sr m^2 s  (assuming Phi in m^-2 s^-1)]

    # Compare unweighted vs atmospheric-weighted energy distribution
    bins = np.logspace(3, 5, 15)
    centers = np.sqrt(bins[:-1] * bins[1:])

    raw_counts,  _ = np.histogram(energies,              bins=bins)
    atm_weighted, _ = np.histogram(energies, bins=bins, weights=atm_weights)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.stairs(raw_counts,  bins, fill=True, color="steelblue", alpha=0.7, label="Unweighted (E^-1 injection)")
    ax.set_xscale("log")
    ax.set_xlabel("Neutrino energy [GeV]")
    ax.set_ylabel("Events")
    ax.set_title("Unweighted distribution")
    ax.legend()

    ax = axes[1]
    ax.stairs(atm_weighted, bins, fill=True, color="darkorange", alpha=0.7, label=r"Reweighted ($E^{-3.7}$ proxy)")
    ax.set_xscale("log")
    ax.set_xlabel("Neutrino energy [GeV]")
    ax.set_ylabel("Weighted event rate [a.u.]")
    ax.set_title("After reweighting to atmospheric flux proxy")
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "reweighted_spectrum.pdf"), bbox_inches="tight")
    plt.show()

    print("Reweighting to a softer spectrum suppresses the high-energy events.")
else:
    print("Skipping reweighting example (weighting not available).")

---
## Section 8: Preparing PPC Tables for a Water Detector

### What do PPC table files do?

PPC (Photon Propagation Code) uses a set of text-format configuration files that completely specify the optical properties of the medium and the detector response:

| File | Purpose |
|---|---|
| `cfg.txt` | Global PPC settings: DOM radius oversize, overall efficiency, scattering asymmetry, and mean cosine g |
| `icemodel.dat` | Layer-by-layer optical properties: depth, effective scattering coefficient b_e(400 nm), absorption coefficient a_dust(400 nm), and temperature correction delta_tau |
| `icemodel.par` | Six parameters for the wavelength-dependent absorption formula (SPICE model): alpha, kappa, A, B, D, E |
| `wv.dat` | Quantum efficiency (QE) of the optical module vs wavelength [nm] |
| `as.dat` | Angular sensitivity of the DOM. A single value of 1 means record all photons (isotropic). |
| `rnd.txt` | Pre-computed random number table for PPC's internal RNG. Should be copied as-is. |
| `tilt.dat` / `tilt.par` | (Optional) Ice-layer tilt corrections. Not used for simple water models. |

### Ice vs Water optical properties

South Pole ice and deep ocean water have very different optical properties:

| Property | South Pole Ice (400 nm) | Mediterranean Water (400 nm) |
|---|---|---|
| Absorption length | ~100 m | ~50 m |
| Effective scattering length | ~20–40 m | ~160 m |
| g = &lt;cos θ&gt; | 0.90 | 0.94 |

Water is much less scattering than ice but has shorter absorption lengths. This gives water detectors a sharper timing response but smaller light yield per unit volume compared to ice.

In [ ]:
import numpy as np

# Helper functions to write each PPC table file

def write_cfg(output_dir, oversize=1, efficiency=1.0, sam_fraction=0.45, g=0.94):
    """
    Write cfg.txt — global PPC configuration.

    Parameters
    ----------
    oversize      : DOM radius oversize scaling factor (1 = no scaling)
    efficiency    : overall DOM efficiency correction multiplier
    sam_fraction  : scattering asymmetry parameter (0=Henyey-Greenstein, 1=SAM)
    g             : mean cosine of scattering angle <cos(theta)>
    """
    path = os.path.join(output_dir, "cfg.txt")
    with open(path, "w") as f:
        f.write(f"{oversize}     # over-R: DOM radius oversize scaling factor\n")
        f.write(f"{efficiency:.1f}   # overall DOM efficiency correction\n")
        f.write(f"{sam_fraction:.2f}  # 0=HG; 1=SAM\n")
        f.write(f"{g:.2f}  # g=<cos(theta)>\n")
    print(f"  Wrote: {path}")


def write_icemodel_par(output_dir, alpha, kappa, A, B, D, E, uncertainties=None):
    """
    Write icemodel.par — SPICE wavelength-dependent absorption parameters.

    The absorption coefficient at wavelength lambda [nm] is:
        a(lambda) = (alpha * a_dust(400) + air * exp(-kappa/lambda)) * (1 + D * delta_tau)
    where air = A * exp(-B/lambda^kappa) describes the bulk absorption.

    Parameters (value, uncertainty) for: alpha, kappa, A, B, D, E
    """
    if uncertainties is None:
        uncertainties = [0.0] * 6
    params = [(alpha, uncertainties[0]), (kappa, uncertainties[1]),
              (A, uncertainties[2]),     (B, uncertainties[3]),
              (D, uncertainties[4]),     (E, uncertainties[5])]
    path = os.path.join(output_dir, "icemodel.par")
    with open(path, "w") as f:
        for val, unc in params:
            f.write(f"    {val:20.12f}   {unc:20.12f}\n")
    print(f"  Wrote: {path}")


def write_icemodel_dat(output_dir, n_layers, depth_min, depth_max,
                       absorption_400, scattering_400, delta_tau=0.0):
    """
    Write icemodel.dat — per-layer optical properties.

    Columns: depth_center [m]  b_e(400) [m^-1]  a_dust(400) [m^-1]  delta_tau

    For a homogeneous medium (e.g., ocean water), all rows are identical.

    Parameters
    ----------
    n_layers       : number of depth layers
    depth_min      : shallowest layer center depth [m, positive downward]
    depth_max      : deepest layer center depth [m]
    absorption_400 : absorption coefficient at 400 nm [m^-1] (= 1 / absorption_length)
    scattering_400 : effective scattering coefficient at 400 nm [m^-1]
    delta_tau      : temperature correction parameter (0 for water)
    """
    step = (depth_max - depth_min) / (n_layers - 1)
    depths = depth_min + np.arange(n_layers) * step
    path   = os.path.join(output_dir, "icemodel.dat")
    with open(path, "w") as f:
        for d in depths:
            f.write(f"{d:.2f} {absorption_400:.6f} {scattering_400:.6f} {delta_tau:.6f}\n")
    print(f"  Wrote: {path}  ({n_layers} layers, depth {depth_min:.0f}–{depth_max:.0f} m)")


def write_wv_dat(output_dir, wavelengths, efficiencies):
    """
    Write wv.dat — quantum efficiency (QE) vs wavelength.

    Parameters
    ----------
    wavelengths  : array of wavelengths [nm], typically 265–675 nm
    efficiencies : array of QE values [0, 1]
    """
    efficiencies = np.array(efficiencies)
    cdf = np.cumsum(efficiencies)
    cdf /= cdf[-1]
    
    # PPC requires strictly increasing CDF — keep first point (must be 0)
    # then only points where CDF strictly increases
    keep = np.concatenate([[True], np.diff(cdf) > 0])
    wavelengths = np.array(wavelengths)[keep]
    cdf = cdf[keep]
    
    # Force first CDF value to exactly 0
    cdf[0] = 0.0
    
    path = os.path.join(output_dir, "wv.dat")
    with open(path, "w") as f:
      for wv, c in zip(wavelengths, cdf):
          f.write(f"{c:.8f} {wv:.0f}.\n")
    print(f"  Wrote: {path}  ({len(wavelengths)} wavelength points)")


def write_as_dat(output_dir):
    """
    Write as.dat — angular sensitivity of the DOM.

    A single value of 1 (isotropic) records all photons regardless of
    their angle of incidence. Use this for generic multi-PMT modules.
    """
    path = os.path.join(output_dir, "as.dat")
    with open(path, "w") as f:
        f.write("1\n")
    print(f"  Wrote: {path}")


print("Table-writing functions defined.")

In [ ]:
# Display the south pole ice model parameters for reference
print("=== South Pole Ice Model Parameters (SPICE) ===")
print()

sp_par = np.genfromtxt(os.path.join(PPC_TABLES_SP, "icemodel.par"))
param_names = ["alpha", "kappa", "A", "B", "D", "E"]
param_descr = [
    "Scattering slope",
    "Absorption wavelength exponent",
    "Absorption amplitude A [1/m]",
    "Absorption B [nm * kappa]",
    "Temperature correction D",
    "Temperature correction E",
]
for name, (val, unc), descr in zip(param_names, sp_par, param_descr):
    print(f"  {name:6s} = {val:12.6f}  +/- {unc:.6f}   ({descr})")

sp_dat = np.genfromtxt(os.path.join(PPC_TABLES_SP, "icemodel.dat"))
print()
print(f"  icemodel.dat: {len(sp_dat)} depth layers")
print(f"  Depth range : {sp_dat[:, 0].min():.0f} – {sp_dat[:, 0].max():.0f} m")
print(f"  b_e(400) range  : {sp_dat[:, 1].min():.4f} – {sp_dat[:, 1].max():.4f} m^-1")
print(f"  a_dust(400) range: {sp_dat[:, 2].min():.4f} – {sp_dat[:, 2].max():.4f} m^-1")

In [ ]:
# Create Mediterranean water tables
#
# Optical properties for ANTARES/KM3NeT site (depth ~2500 m):
#   Absorption length at 400 nm : ~50 m  -> a_400 = 0.020 m^-1
#   Scattering length at 400 nm : ~160 m -> b_e_400 = 0.006 m^-1
# (effective scattering after accounting for g ~ 0.94)

WATER_TABLES_DIR = os.path.join(OUTPUT_DIR, "water_tables")
os.makedirs(WATER_TABLES_DIR, exist_ok=True)
print(f"Creating Mediterranean water PPC tables in: {WATER_TABLES_DIR}")
print()

# cfg.txt: water has higher g than ice (~0.94 vs 0.90)
write_cfg(
    WATER_TABLES_DIR,
    oversize=1,
    efficiency=1.0,
    sam_fraction=0.45,
    g=0.94
)

# icemodel.par: simplified water absorption parameters
# a(lambda) = A * exp(-B/lambda^kappa)  (dust-free deep water)
shutil.copy(os.path.join(PPC_TABLES_SP, "icemodel.par"), os.path.join(WATER_TABLES_DIR, "icemodel.par"))

# Solve for a_dust to get desired total absorption at 400 nm
# PPC formula: abs = (D * a_dust + E) * 400^(-kappa) + A * exp(-B/400)
sp_par_vals = np.genfromtxt(os.path.join(PPC_TABLES_SP, "icemodel.par"))[:,0]
_, kappa_sp, A_sp, B_sp, D_sp, E_sp = sp_par_vals
l_k_400 = 400.0**(-kappa_sp)
ABl_400 = A_sp * np.exp(-B_sp / 400.0)
target_absorption = 1/50.0  # 50 m absorption length -> 0.020 m^-1
a_dust_water = (target_absorption - E_sp*l_k_400 - ABl_400) / (D_sp * l_k_400)
print(f"  a_dust for 50m absorption length: {a_dust_water:.6f} m^-1")
 

# icemodel.dat: 100 uniform layers from 2000 m to 3000 m depth
# Mediterranean site: ANTARES at ~2475 m, KM3NeT/ORCA at 2450 m
write_icemodel_dat(
    WATER_TABLES_DIR,
    n_layers=101,
    depth_min=2000.0,
    depth_max=3000.0,
    absorption_400=a_dust_water,
    scattering_400=0.00625,
    delta_tau=0.0
)

# wv.dat: generic multi-PMT QE peaking around 420 nm
# KM3NeT uses 3" Hamamatsu PMTs with QE peak ~0.32 at ~420 nm.
# We use a simpler Gaussian-shaped model peaking at 420 nm.
wv_wavelengths  = np.arange(265, 685, 10, dtype=float)
peak_wv         = 420.0
sigma_wv        = 80.0
peak_qe         = 0.25
wv_efficiencies = peak_qe * np.exp(-0.5 * ((wv_wavelengths - peak_wv) / sigma_wv) ** 2)
# Ensure QE is zero below ~280 nm (PMT glass cutoff)
wv_efficiencies[wv_wavelengths < 280] = 0.0
# Normalise so max = peak_qe
wv_efficiencies = wv_efficiencies / wv_efficiencies.max() * peak_qe

write_wv_dat(WATER_TABLES_DIR, wv_wavelengths, wv_efficiencies)

# as.dat: isotropic angular sensitivity
write_as_dat(WATER_TABLES_DIR)

# Copy rnd.txt from south_pole (random number table)
shutil.copy(
    os.path.join(PPC_TABLES_SP, "rnd.txt"),
    os.path.join(WATER_TABLES_DIR, "rnd.txt")
)
print(f"  Copied: rnd.txt")
shutil.copy(os.path.join(PPC_TABLES_SP, "tilt.dat"), os.path.join(WATER_TABLES_DIR, "tilt.dat"))
shutil.copy(os.path.join(PPC_TABLES_SP, "tilt.par"), os.path.join(WATER_TABLES_DIR, "tilt.par"))

print()
print("Water table files created:")
for f in sorted(os.listdir(WATER_TABLES_DIR)):
    fpath = os.path.join(WATER_TABLES_DIR, f)
    print(f"  {f:20s}  ({os.path.getsize(fpath):6d} bytes)")

In [ ]:
# Compare ice vs water optical properties graphically

# Load south pole data
sp_dat = np.genfromtxt(os.path.join(PPC_TABLES_SP, "icemodel.dat"))
sp_depths   = sp_dat[:, 0]  # m
sp_be400    = sp_dat[:, 1]  # effective scattering [m^-1]
sp_adust400 = sp_dat[:, 2]  # absorption (dust component) [m^-1]

# Load south pole QE
sp_wv_data  = np.genfromtxt(os.path.join(PPC_TABLES_SP, "wv.dat"))
sp_wv_eff   = sp_wv_data[:, 0]
sp_wv_nm    = sp_wv_data[:, 1]

# Load water QE
water_wv_data = np.genfromtxt(os.path.join(WATER_TABLES_DIR, "wv.dat"))
water_wv_eff  = water_wv_data[:, 0]
water_wv_nm   = water_wv_data[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# SPICE model parameters
alpha, kappa, A, B, D, E = np.genfromtxt(os.path.join(PPC_TABLES_SP, "icemodel.par"))[:, 0]

wva = 400.0  # reference wavelength [nm]
l_k = wva ** (-kappa)
ABl = A * np.exp(-B / wva)  # pure ice contribution at 400 nm

# Full SPICE absorption: dust + pure ice + temperature correction
a_total = (D * sp_adust400 + E) * l_k + ABl * (1 + 0.01 * sp_dat[:, 3])
valid = (sp_adust400 > 0) & (sp_adust400 < 10)

# 1. Absorption length vs depth (ice)
ax = axes[0, 0]
# Avoid division by zero
valid = (sp_adust400 > 0) & (sp_adust400 < 10)
ax.plot(a_total[valid], sp_depths[valid], color="steelblue", lw=1.5)
ax.axhline(0, color="k", linestyle="--", alpha=0.3, label="Sea level")
ax.set_xlabel("Absorption length at 400 nm [m]")
ax.set_ylabel("Depth [m]")
ax.set_title("South Pole Ice: absorption length vs depth")
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# 2. Effective scattering length vs depth (ice)
ax = axes[0, 1]
valid_sc = (sp_be400 > 0) & (sp_be400 < 10)
ax.axhline(0, color="k", linestyle="--", alpha=0.3, label="Sea level")
ax.plot(1.0 / sp_be400[valid_sc], sp_depths[valid_sc], color="firebrick", lw=1.5)
ax.set_xlabel("Effective scattering length at 400 nm [m]")
ax.set_ylabel("Depth [m]")
ax.set_title("South Pole Ice: scattering length vs depth")
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# 3. QE comparison: IceCube DOM vs generic water multi-PMT
ax = axes[1, 0]
ax.plot(sp_wv_nm,    sp_wv_eff,    color="steelblue",  lw=2, label="IceCube DOM (south_pole)")
ax.plot(water_wv_nm, water_wv_eff, color="darkorange", lw=2, label="Generic water multi-PMT (model)")
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Quantum efficiency")
ax.set_title("Quantum efficiency vs wavelength")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Summary bar chart of absorption and scattering at 400 nm
ax = axes[1, 1]
media   = ["South Pole\nIce (mean)", "Mediterranean\nWater"]
abs_len = [1.0 / a_total[valid].mean(), 50.0]
sca_len = [1.0 / sp_be400[valid_sc].mean(), 160.0]

x = np.arange(len(media))
w = 0.35
bars1 = ax.bar(x - w/2, abs_len, w, label="Absorption length", color="steelblue")
bars2 = ax.bar(x + w/2, sca_len, w, label="Scattering length",  color="firebrick")
ax.bar_label(bars1, fmt="%.0f m", padding=3)
ax.bar_label(bars2, fmt="%.0f m", padding=3)
ax.set_xticks(x)
ax.set_xticklabels(media)
ax.set_ylabel("Optical length at 400 nm [m]")
ax.set_title("Ice vs water optical properties at 400 nm")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "optical_properties_comparison.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Run a simulation using the custom water optical tables.
#
# NOTE: we use demo_ice.geo (an ice-medium geometry) with water optical tables
# deliberately — this tests the table-loading mechanism without requiring a
# separate water-detector geometry file. For a realistic water simulation,
# use orca.geo or arca.geo with medium='water'.

import importlib
import prometheus as prom_module
importlib.reload(prom_module)
from prometheus import Prometheus, config

_WATER_OUTFILE = os.path.join(OUTPUT_DIR, "my_water.parquet")

if os.path.exists(_WATER_OUTFILE):
    print(f"Pre-computed output found: {_WATER_OUTFILE}")
    print("Skipping simulation — delete the file to re-run.")
else:
    config["run"]["nevents"]           = 50
    config["run"]["random state seed"] = 77
    config["run"]["outfile"]          = _WATER_OUTFILE
    config["run"]["storage prefix"]   = OUTPUT_DIR

    config["detector"]["geo file"] = GEO_FILE_ICE

    config["injection"]["LeptonInjector"]["paths"]["xsec dir"]               = XSEC_DIR
    config["injection"]["LeptonInjector"]["simulation"]["final state 1"]     = "MuMinus"
    config["injection"]["LeptonInjector"]["simulation"]["final state 2"]     = "Hadrons"
    config["injection"]["LeptonInjector"]["simulation"]["minimal energy"]    = 1e3
    config["injection"]["LeptonInjector"]["simulation"]["maximal energy"]    = 1e4

    config["photon propagator"]["name"]                               = "PPC"
    config["photon propagator"]["PPC"]["paths"]["ppc_exe"]            = PPC_EXE
    config["photon propagator"]["PPC"]["paths"]["ppctables"]          = WATER_TABLES_DIR
    config["photon propagator"]["PPC"]["paths"]["ppc_tmpdir"]         = PPC_TMPDIR
    config["photon propagator"]["PPC"]["paths"]["force"]              = True
    print(f"Running 50-event simulation with custom water optical tables...")
    p_water = Prometheus(userconfig=config)
    p_water.sim()
    print(f"Water simulation complete.")

print(f"Output: {_WATER_OUTFILE}")

# Use simulation output if available, otherwise fall back to pre-computed
_my_water = os.path.join(OUTPUT_DIR, "my_water.parquet")
_precomputed_water = os.path.join(OUTPUT_DIR, "tutorial_water.parquet")
WATER_PARQUET = _my_water if os.path.exists(_my_water) else _precomputed_water
print(f"WATER_PARQUET -> {WATER_PARQUET}")

In [ ]:
# Compare hit statistics: ice tables vs water tables (same geometry and events)
# Per-event means are compared since the two samples may have different event counts.

data_ice_tables   = ak.from_parquet(NUE_PARQUET)
data_water_tables = ak.from_parquet(WATER_PARQUET)

nhits_ice_tbl   = ak.num(data_ice_tables["photons"]["t"]).to_numpy()
nhits_water_tbl = ak.num(data_water_tables["photons"]["t"]).to_numpy()

print("=== Hit statistics comparison ===")
print()
print(f"Ice tables (south_pole, {len(data_ice_tables)} events):")
print(f"  Mean hits per event    : {nhits_ice_tbl.mean():.1f}")
print(f"  Fraction with hits     : {(nhits_ice_tbl > 0).mean():.2f}")
print()
print(f"Water tables (custom, 10 events):")
print(f"  Mean hits per event    : {nhits_water_tbl.mean():.1f}")
print(f"  Fraction with hits     : {(nhits_water_tbl > 0).mean():.2f}")
print()
print("Differences are expected: the water tables have shorter absorption lengths")
print("and weaker scattering, giving different photon collection efficiency.")
print()
print("The goal here was to demonstrate table-loading — not physically accurate water simulation.")
print("For a real study, use the correct water geometry (orca.geo / arca.geo).")

---
## Summary

This tutorial has covered the full Prometheus workflow:

1. **Setup**: Defining paths using the package location so the notebook is portable.

2. **Running a simulation**: Configuring and running a 50-event muon-neutrino CC simulation on a small ice detector with PPC photon propagation.

3. **Output inspection**: Loading the Parquet output with awkward-array, exploring the schema, reading MC truth fields, and histogramming hit multiplicities.

4. **Event visualization**: Using `plot_brightest` for 3-D event displays, and making custom arrival-time and hit-position plots.

5. **MC truth distributions**: Energy spectrum, zenith distribution, vertex positions, Bjorken kinematics, and final-state particle breakdown.

6. **Effective area**: Loading oneweights with `ParquetWeighter`, applying a hit-multiplicity cut, and computing $A_\text{eff}(E)$.

7. **Custom physics**: Reweighting to an atmospheric flux proxy, and running a separate electron-neutrino (cascade) simulation for comparison, illustrating the track vs cascade topology difference.

8. **Custom PPC tables**: Writing all PPC table files from scratch for Mediterranean deep water, visualizing the optical property differences, and running a test simulation with the new tables.

### Next steps

- **Larger statistics**: Increase `nevents` to 10,000+ for statistically meaningful distributions. Use `fasrc_example_ppc.py` as a template for cluster submission.
- **Water detectors**: Replace `demo_ice.geo` with `orca.geo` or `arca.geo` and set `medium=water` to use the Olympus photon propagator, which is designed for water Cherenkov detectors.
- **Custom geometries**: Use `detector_from_geo` and `Detector.to_geo` to create and simulate novel detector geometries.
- **Flux models**: Replace the power-law proxy in Section 7 with a physics-motivated flux from the `MCEq` or `nuflux` packages.
- **Neural networks**: Prometheus output can serve as training data for deep-learning event reconstruction algorithms.